# Scenarios quickstart

This notebook shows how to build, compose, and apply scenarios using the Finstack Python bindings. We'll create simple market shocks, compose them into a single scenario, apply them to markets/statements, and serialize them to JSON.


In [53]:
from datetime import date

from finstack import Currency, Money
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.scenarios import (
    CurveKind,
    ExecutionContext,
    OperationSpec,
    ScenarioEngine,
    ScenarioSpec,
    TenorMatchMode,
    VolSurfaceKind,
)
from finstack.statements.types import FinancialModelSpec


## Setup helpers


In [54]:
base_date = date(2025, 1, 1)

# Simple USD discount curve knots (time in years, discount factor)
knots = [(0.0, 1.0), (1.0, 0.98), (5.0, 0.90), (10.0, 0.80)]


def make_market() -> MarketContext:
    """Create a fresh market context with a discount curve and equity prices."""
    market = MarketContext()
    market.insert_discount(DiscountCurve("USD-OIS", base_date, knots))
    market.insert_price("SPY", MarketScalar.price(Money(450.0, Currency("USD"))))
    market.insert_price("QQQ", MarketScalar.price(Money(380.0, Currency("USD"))))
    return market


def baseline_df(years: float = 1.0) -> float:
    """Convenience helper for the original discount factor."""
    return DiscountCurve("USD-OIS", base_date, knots).df(years)



## Create a baseline market and model


In [55]:
market = make_market()
model = FinancialModelSpec("demo_model", [])

print("Market created with:")
print("- Discount curves: ['USD-OIS']")
print("- Equities: ['SPY', 'QQQ']")
print(f"Baseline 1Y DF: {baseline_df():.6f}")


Market created with:
- Discount curves: ['USD-OIS']
- Equities: ['SPY', 'QQQ']
Baseline 1Y DF: 0.980000


## Define scenarios

We will create a few reusable scenarios to demonstrate curve shocks, equity shocks, and tenor-specific moves.


In [56]:
rate_shock = ScenarioSpec(
    "rate_shock",
    [OperationSpec.curve_parallel_bp(CurveKind.Discount, "USD-OIS", 50.0)],
    name="Rate Shock +50bp",
    description="Parallel shift of USD discount curve",
    priority=0,
)

equity_crash = ScenarioSpec(
    "equity_crash",
    [OperationSpec.equity_price_pct(["SPY", "QQQ"], -20.0)],
    name="Equity Crash -20%",
    description="Market sell-off scenario",
    priority=1,
)

combined_stress = ScenarioSpec(
    "combined_stress",
    [
        OperationSpec.curve_parallel_bp(CurveKind.Discount, "USD-OIS", 100.0),
        OperationSpec.equity_price_pct(["SPY", "QQQ"], -30.0),
    ],
    name="Combined Stress",
    description="Rates up 100bp + Equities down 30%",
    priority=2,
)

node_shock = ScenarioSpec(
    "steepener",
    [
        OperationSpec.curve_node_bp(
            CurveKind.Discount,
            "USD-OIS",
            [("5Y", 25.0), ("10Y", 50.0)],
            TenorMatchMode.Interpolate,
        )
    ],
    name="Curve Steepener",
    description="Targeted moves on the 5Y/10Y nodes",
)

print("Defined scenarios:")
for s in [rate_shock, equity_crash, combined_stress, node_shock]:
    print(f"- {s.id}: {s.name} (ops={len(s.operations)})")


Defined scenarios:
- rate_shock: Rate Shock +50bp (ops=1)
- equity_crash: Equity Crash -20% (ops=1)
- combined_stress: Combined Stress (ops=2)
- steepener: Curve Steepener (ops=1)


## Compose scenarios


In [57]:
engine = ScenarioEngine()

composed = engine.compose([rate_shock, equity_crash, combined_stress])
print(f"Composed {len(composed.operations)} operations from 3 scenarios")
print(f"Composed scenario ID: {composed.id}")

for i, op in enumerate(composed.operations, 1):
    print(f"  {i:02d}. {op}")


Composed 4 operations from 3 scenarios
Composed scenario ID: composed
  01. CurveParallelBp { curve_kind: Discount, curve_id: "USD-OIS", bp: 50.0 }
  02. EquityPricePct { ids: ["SPY", "QQQ"], pct: -20.0 }
  03. CurveParallelBp { curve_kind: Discount, curve_id: "USD-OIS", bp: 100.0 }
  04. EquityPricePct { ids: ["SPY", "QQQ"], pct: -30.0 }


## Apply a rate shock


In [58]:
market_rate = make_market()
ctx = ExecutionContext(market_rate, model, base_date)

report = engine.apply(rate_shock, ctx)
shocked_curve = market_rate.discount("USD-OIS")
shocked_df = shocked_curve.df(1.0)

print(f"Operations applied: {report.operations_applied}")
print(f"Baseline 1Y DF: {baseline_df():.6f}")
print(f"Shocked  1Y DF: {shocked_df:.6f}")
print(f"Change (bps): {(shocked_df - baseline_df()) * 10000:.2f}")

if report.warnings:
    print("Warnings:")
    for w in report.warnings:
        print(f"- {w}")


Operations applied: 1
Baseline 1Y DF: 0.980000
Shocked  1Y DF: 0.974888
Change (bps): -51.12


## Apply an equity crash


In [59]:
market_equity = make_market()
ctx_eq = ExecutionContext(market_equity, model, base_date)
report_eq = engine.apply(equity_crash, ctx_eq)

shocked_spy = market_equity.price("SPY")

print(f"Operations applied: {report_eq.operations_applied}")
print(f"SPY before: $450.00")
print(f"SPY after : ${shocked_spy.value.amount:.2f}")


Operations applied: 2
SPY before: $450.00
SPY after : $360.00


## Serialize to/from JSON


In [60]:
json_str = rate_shock.to_json()
rehydrated = ScenarioSpec.from_json(json_str)

print(f"Serialized length: {len(json_str)} chars")
print(f"Round-tripped ID: {rehydrated.id}")
print(f"Operations recovered: {len(rehydrated.operations)}")


Serialized length: 276 chars
Round-tripped ID: rate_shock
Operations recovered: 1


## Tenor-specific curve shocks


In [61]:
market_nodes = make_market()
ctx_nodes = ExecutionContext(market_nodes, model, base_date)
report_nodes = engine.apply(node_shock, ctx_nodes)

curve = market_nodes.discount("USD-OIS")
points = [1.0, 5.0, 10.0]

print(f"Operations applied: {report_nodes.operations_applied}")
print("Shocked curve discount factors:")
for t in points:
    shocked = curve.df(t)
    base = baseline_df(t)
    print(f"  t={t:>4.1f}y -> df={shocked:.6f} (delta bps: {(shocked - base) * 10000:.2f})")


Operations applied: 1
Shocked curve discount factors:
  t= 1.0y -> df=0.979729 (delta bps: -2.71)
  t= 5.0y -> df=0.888705 (delta bps: -112.95)
  t=10.0y -> df=0.766962 (delta bps: -330.38)


## Next steps

- Combine scenario application with valuations or statements to see P&L and liquidity impacts.
- Expand `OperationSpec` usage (vol surfaces, FX curves) and adjust priorities to control ordering.
- Use `ScenarioEngine.compose` to build reusable bundles for regression and stress testing.
